# 内存结构

Polars 的 DataFrame 背后是 Apache Arrow 格式的数组，所以内存效率高、数据访问也快。Apache Arrow 是专为数据分析设计的列式内存格式，特点有四个。

1. 内存效率：数据存放在连续的内存块里，对缓存友好，处理大量数据时占用也能压到最低。

2. 兼容性：Arrow 格式和许多数据分析工具兼容，数据在不同工具之间搬动很顺畅，不必反复序列化。

3. 快速访问：列式存储让「只取某几列」非常快，数据量大、又反复操作某几列时尤其明显。

4. 数据类型丰富：Arrow 支持 NULL、列表、结构体、字符串等类型，复杂结构也能直接表达。

这一章就讲 NULL、字符串、Array、List、Struct 这几种类型的内存结构。看懂它，你就能明白为什么某些操作是「零拷贝」、为什么改一个值有时会复制整列。

In [1]:
import struct
import polars as pl

## 数据的管理

Polars 最底层的数据管理用的是 `pyarrow` 的 `Buffer`。理解这一层的关键是分清「对象」和「内存」：多个对象可以指向同一块内存。

下面的代码把 `df['A']` 取两次，用 `id()` 比较这两个 `Series` 对象是否相同。

In [2]:
df = pl.DataFrame({
    "A": [1, 2, 3, 4],
    "B": [5, 6, 7, 8]
})

# 取两次 Series 对象
s1 = df['A']
s2 = df['A']
print(id(s1), id(s2))

2394256136528 2394225185616


结果是取到的两个 `Series` 对象**并不相同**。Polars 的 `Series` 内部是 Rust 实现的 `PySeries`，而这两个 `PySeries` 也是不同的对象。

也就是说，每次取列都会新建一层 Python 包装，但包装底下可能还是同一份数据。

In [3]:
print(id(s1._s), id(s2._s))

2394247967776 2394255934208


把每个 `Series` 对象转成 PyArrow 数组，拿到的数组也**并不相同**——又是一层新的包装对象。

In [4]:
a1 = s1.to_arrow()
a2 = s2.to_arrow()
print(id(a1), id(a2))

2394226741152 2394256919520


再取 PyArrow 数组内部使用的 `Buffer` 对象：ID 不同，说明 `Buffer` 对象本身也不同，但用 `address` 属性看数据地址，会发现**地址是同一个**。

所以链条是「对象各不相同，底层内存共用」，下图把这几层对象的关系画了出来。

In [5]:
b1 = a1.buffers()[1]
b2 = a2.buffers()[1]
print(id(b1), id(b2))
print(b1.address, b2.address)

2394264064688 2394264064752
2392299604992 2392299604992


接着用索引操作改动 `Series` 的元素，再确认 `Buffer` 的数据地址。这一步要看的是：修改会不会影响别的对象。

In [6]:
s1[1] = 100
print(s1.to_arrow().buffers()[1].address)

2392299604352


结果数据的地址**变了**。也就是说，这个操作不是在同一地址上覆写，而是把数据存到新地址。

因此 `s2` 和 `df` 还持有原来的数据，看不到 `s1` 的改动——这就是「写时复制」（copy-on-write）的实际表现，也是 Polars 敢把列共享给多个对象的原因。

## NULL 的存储方式

Polars 用 PyArrow 的数据类型保存数据。含 NULL 的数组用两个 buffer（缓冲区）来存：一个记「哪个位置是 NULL」，一个存实际数据。

* **mask buffer**（掩码缓冲）— 逐位记录各元素是否为 NULL：位为 1 表示非 NULL，为 0 表示 NULL。一个元素只占一位，所以数据量再大，NULL 信息也占不了多少。

* **data buffer**（数据缓冲）— 存放实际数据，NULL 的位置用 0 或其它占位值填上，真正有值的元素连续存放。

下面看具体例子。

In [7]:
s = pl.Series([1, None, None, 3, 4, None, 8, 9], dtype=pl.Int16)
a = s.to_arrow()
buf_mask, buf_data = a.buffers()

`buf_mask` 用位串表示 NULL 标记：`0b11011001` 的每一位对应一个元素，从最右一位（第 1 个元素）起依次是 `1, 0, 0, 1, 1, 0, 1, 1`。

可见第 2、3、6 个元素是 NULL。位串是从右往左读的，这一点第一次看很容易搞反。

In [8]:
bin(bytes(buf_mask)[0])

'0b11011001'

In [9]:
struct.unpack('<8h', buf_data)

(1, 0, 0, 3, 4, 0, 8, 9)

下面的代码确认 `buf_data` 和 `s` 共享同一块内存：用 `ctypes` 把 `buf_data.address` 转成指针 `pbuf`，通过它把第 4 个值改成 99，然后打印 `s` 看改动是否生效。

如果 `s` 里也变成 99，就说明两个对象确实指向同一块内存——这就是「零拷贝」的直接证据。

In [10]:
from ctypes import POINTER, c_int16, cast
pbuf = cast(buf_data.address, POINTER(c_int16))
pbuf[3] = 99
print(s)

shape: (8,)
Series: '' [i16]
[
	1
	null
	null
	99
	4
	null
	8
	9
]


## String

Polars 的字符串列用三个 buffer 存数据：不存变长的对象指针，而是把内容连续排在一起、用索引切分。这样字符串的访问和操作都很快。

**mask buffer**（掩码缓冲）— 逐位记录各字符串元素是否为 NULL：位为 1 表示非 NULL，为 0 表示 NULL。

**index buffer**（索引缓冲）— 保存每个字符串元素的起始索引，第 i 个字符串的范围是 `buf_index[i]` 到 `buf_index[i+1]`，从连续数据里一减就能切出来。

**data buffer**（数据缓冲）— 所有字符串连续存放在这里，内存占用压到最小。

下面看具体例子。

In [11]:
s = pl.Series(["abc", "defghi", "xyz", None, "123"])
a = s.to_arrow()
buf_mask, buf_index, buf_data = a.buffers()

`buf_mask` 的每一位表示对应元素的 NULL 状态：前三个元素不是 NULL，第 4 个元素是 NULL。

In [12]:
bin(bytes(buf_mask)[0])

'0b11110111'

`buf_index` 里存的是 `[0, 3, 9, 12, 12, 15]`，也就是各元素的起止索引。

* "abc" 的起始索引是 0，结束索引是 3
* "defghi" 的起始索引是 3，结束索引是 9
* 该元素是 NULL，起止索引都是 12

注意 NULL 那一段的长度是 0：它没有内容，所以在数据缓冲里不占位置。

In [13]:
struct.unpack('<6q', buf_index)

(0, 3, 9, 12, 12, 15)

`buf_data` 里所有字符串数据连续存放；借索引缓冲，就能从这段连续数据里快速切出各元素，而不必为每个字符串单独分配一块内存。

In [14]:
buf_data.to_pybytes()

b'abcdefghixyz123'

比如 `s[2]` 的起始索引是 9、结束索引是 12，从 `buf_data` 里取出的就是这一段：

In [15]:
buf_data[9:12].to_pybytes()

b'xyz'

## Array

Polars 的 Array 列存定长列表：每个元素长度相同，这一点让它的内存结构比 List 简单。

`list_size` 属性尤其关键——它记录每个元素的长度，有了它就不需要索引缓冲。下面用代码例子说明。

In [16]:
s = pl.Series([
    [1,   None, 3   ], 
    [4,   5,    None], 
    [6,   7,    8   ], 
    [9,   10,   11  ]], 
    dtype=pl.Array(pl.Int16, 3))
a = s.to_arrow()

`list_size` 属性表示每个元素的长度，所有元素都一样长，所以只需一个数就能算出每个元素在缓冲里的位置。

In [17]:
print(f"{a.type.list_size=}")

a.type.list_size=3


`buf_mask` 和 `buf_data` 分别存 NULL 和数据，这两个 buffer 和一般列一样；区别在于数据是按「元素 × list_size」排布的。

In [18]:
_, buf_mask, buf_data = a.buffers()
bin(bytes(buf_mask)[0])

'0b11011101'

In [19]:
struct.unpack('<12h', buf_data)

(1, 0, 3, 4, 5, 0, 6, 7, 8, 9, 10, 11)

(list_memory)=
## List

Polars 的 List 列存变长列表：每个元素长度不同，所以除了数据和掩码，还需要一个索引缓冲来记录边界。

这个结构和字符串列非常像——都是「连续数据 + 索引切分」，理解了 String 再看 List 会轻松很多。下面结合代码和输出说明。

In [20]:
s = pl.Series([
    [1, None, 3], 
    [10, 20], 
    None, 
    [100, 200, 300]], 
    dtype=pl.List(pl.Int16))
a = s.to_arrow()
buf_mask1, buf_index, buf_mask2, buf_data = a.buffers()

`buf_mask1` 逐位记录每个列表元素是否为 NULL：位为 1 表示非 NULL，为 0 表示 NULL。第 3 位是 0，说明第 3 个元素是 NULL。

In [21]:
bin(bytes(buf_mask1)[0])

'0b11111011'

`buf_index` 记录每个列表元素的起止索引，和字符串列一样，用它从连续数据里切出各元素。

`s[0]` 的数据索引是 0~3，对应 `buf_data` 里的 `[1, 0, 3]`；`s[1]` 的数据索引是 3~5，对应 `[10, 20]`。

In [22]:
struct.unpack('<5Q', buf_index)

(0, 3, 5, 5, 8)

还有一个 buffer 逐位记录列表内每个元素是否为 NULL。第 2 位是 0，说明第 2 个元素（`None`）是 NULL——这是「NULL 的 NULL」，和「元素本身是空列表」不是一回事。

In [23]:
bin(bytes(buf_mask2)[0])

'0b11111101'

所有列表元素的实际数据连续存在 `buf_data` 里，NULL 值用占位值（本例中是 0）表示，真正有没有值由内层掩码决定。

In [24]:
struct.unpack('<8h', buf_data)

(1, 0, 3, 10, 20, 100, 200, 300)

## Struct

Struct 列存带多个字段的数据：每个字段有自己的类型和内存结构，外层再加一层掩码记录「整行是否为 NULL」。

下面用代码例子说明 Struct 列的内存结构。

In [25]:
s = pl.Series([{"A": 1, "B": None}, {"A": None, "B": 20}, {"A": 3, "B": 30}, None])
a = s.to_arrow()
a.buffers()

[<pyarrow.Buffer address=0x22d00210a50 size=1 is_cpu=True is_mutable=False>,
 <pyarrow.Buffer address=0x22d00210a48 size=1 is_cpu=True is_mutable=False>,
 <pyarrow.Buffer address=0x22d002b04e0 size=32 is_cpu=True is_mutable=False>,
 <pyarrow.Buffer address=0x22d00210a40 size=1 is_cpu=True is_mutable=False>,
 <pyarrow.Buffer address=0x22d002b04a0 size=32 is_cpu=True is_mutable=False>]

一个字段用两个 buffer：一个存掩码，一个存值。从这种内存结构可以看出，各字段的值在内存里是连续存放的——这也是 Struct 比「多张表 join」更省内存的原因。

In [26]:
buf_mask_A, buf_value_A = a.buffers()[1:3]
print(bin(bytes(buf_mask_A)[0]))
print(struct.unpack('<4Q', buf_value_A))

0b101
(1, 0, 3, 0)


In [27]:
buf_mask_B, buf_value_B = a.buffers()[3:5]
print(bin(bytes(buf_mask_B)[0]))
print(struct.unpack('<4Q', buf_value_B))

0b110
(0, 20, 30, 0)
